# G1 Academy Bonus - Task 6: basic locomotion with the wrapper

## Introduction
You drive the robot through the finished `sdk_wrapper.G1` wrapper -- no native `LocoClient`, no DDS subscriber classes. `g1.loco_move(vx, vy, vyaw, duration_s)` is the one call this whole task is about: a timed velocity command that auto-stops for you. You will read odometry, drive a few timed segments, and assemble them into a simple multi-stop "U" path that Day 3 reuses (adding an arm gesture at each stop).

**Using Codex/AI for this task:** `loco_move`/`loco_stop`/`get_odom` are finished, documented methods on `sdk_wrapper.G1`. If a cell is not obvious, paste the signature (or the matching line from `wrapper_cheatsheet.html`) into Codex, then read what it produced before you run it against the robot.

In [ ]:
import sys
import time
sys.path.append("..")
from sdk_wrapper import G1

g1 = G1(iface="eth0", domain_id=0)
g1.damp_mode(); g1.prepare_mode(); g1.walk_mode()  # always step up through the modes before moving

## Task 1 - Read odometry: `get_odom()`
`g1.get_odom()` is your ground truth for how far the robot actually went -- open-loop timing drifts, so read it before and after each move rather than trusting the commanded velocity × duration.

In [ ]:
print(g1.get_odom())

## Task 2 - `g1.loco_move(vx, vy, vyaw, duration_s)`
- `loco_move(vx, vy, vyaw, duration_s=2)` drives for exactly `duration_s` seconds, then auto-stops (calls `loco_stop()` for you) -- the common case.
- `loco_move(vx, vy, vyaw, duration_s=None)` fires one `Move()` RPC and returns immediately, no auto-stop -- only use this when you manage the stop yourself.
- `loco_stop()` stops immediately -- call any time you need to bail out of a move early.

`vx`/`vy` are m/s (forward/strafe), `vyaw` is rad/s. Start small (≈0.15 m/s) with a spotter in a clear space.

In [ ]:
before = g1.get_odom()
g1.loco_move(vx=0.2, vy=0, vyaw=0, duration_s=2)
after = g1.get_odom()
print(before, "->", after)

## Task 3 - a timed multi-stop path ("U" path)
A path is just several `loco_move(...)` calls back to back -- forward, turn, forward, turn, forward traces an approximate "U" in 3 legs. Keep the stop points as data and let a small `on_stop(g1, stop_index)` callback run at each stop: that is exactly the hook Day 3's Task 1 reuses to play a different arm gesture at each stop of the *same* U path.

In [ ]:
def walk_u_path(g1, vx=0.15, vyaw=0.5, leg_s=2.0, turn_s=1.2, pause_s=1.0, on_stop=None):
    """Drives an approximate U path as 3 timed legs with a turn between each.
    Calls on_stop(g1, stop_index) at each of the 3 stop points, if given -- this is
    the hook Day 3 Task 1 uses to play a different arm gesture at each stop."""
    legs = [(vx, 0.0, 0.0, leg_s), (0.0, 0.0, vyaw, turn_s),
            (vx, 0.0, 0.0, leg_s), (0.0, 0.0, vyaw, turn_s),
            (vx, 0.0, 0.0, leg_s)]
    stop_index = 0
    for i, (lvx, lvy, lvyaw, duration_s) in enumerate(legs):
        g1.loco_move(lvx, lvy, lvyaw, duration_s=duration_s)
        if i % 2 == 0:  # after each straight leg, we are at a stop point
            if on_stop is not None:
                on_stop(g1, stop_index)
            time.sleep(pause_s)
            stop_index += 1
    g1.loco_stop()

walk_u_path(g1)

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.